In [ ]:
import torch
import numpy as np
import twixtools

def import_kspace(filename):
    """
    Reads Siemens .dat file using twixtools, extracts imaging k-space data,
    and returns both the data array and acquisition info.

    Returns:
        raw_file : full Twix object
        kspace   : np.ndarray [n_slice, n_part, n_line, n_channel, n_column]
        info     : dict with metadata and detected acquisition type
    """
    # --- Read the raw file ---
    raw_file = twixtools.read_twix(filename)[-1]
    mdb_list = raw_file["mdb"]

    # --- Keep only imaging scans ---
    image_mdbs = [mdb for mdb in mdb_list if mdb.is_image_scan()]
    if not image_mdbs:
        raise RuntimeError("No imaging scans found in this .dat file!")

    # --- Infer dimensions from counters ---
    n_slice = 1 + max(mdb.cSlc for mdb in image_mdbs)
    n_part  = 1 + max(mdb.cPar for mdb in image_mdbs)
    n_line  = 1 + max(mdb.cLin for mdb in image_mdbs)
    n_channel, n_column = image_mdbs[0].data.shape

    # --- Allocate k-space tensor ---
    kspace = np.zeros((n_slice, n_part, n_line, n_channel, n_column), dtype=np.complex64)

    # --- Fill in data properly ---
    for mdb in image_mdbs:
        cSlc, cPar, cLin = mdb.cSlc, mdb.cPar, mdb.cLin
        kspace[cSlc, cPar, cLin] += mdb.data  # sum in case of averages

    # --- Detect acquisition type ---
    unique_slices = sorted({mdb.cSlc for mdb in image_mdbs})
    unique_parts  = sorted({mdb.cPar for mdb in image_mdbs})
    n_slices = len(unique_slices)
    n_parts  = len(unique_parts)

    if n_parts > 1:
        acq_type = "3D acquisition (multi-partition)"
    elif n_slices > 1:
        acq_type = "2D multi-slice acquisition"
    else:
        acq_type = "2D single-slice acquisition"

    # --- Print summary ---
    print(f"\nFile: {filename}")
    print(f"K-space dimensions: slices={n_slice}, partitions={n_part}, "
          f"lines={n_line}, channels={n_channel}, columns={n_column}")
    print(f"Detected: {acq_type}\n")
    print(f"Readout Oversampling Factor: {raw_file['hdr']['Config']['ReadoutOversamplingFactor']}")

    # --- Metadata dictionary ---
    info = {
        "filename": filename,
        "dims": (n_slice, n_part, n_line, n_channel, n_column),
        "acquisition_type": acq_type,
        "n_slices": n_slices,
        "n_partitions": n_parts,
        "readout_oversampling_factor": raw_file['hdr']['Config']['ReadoutOversamplingFactor'],
        "description": raw_file['hdr']['Config']['tStudyDescription']
    }

    return raw_file, kspace, info

filename = "/Users/berktinaz/Documents/Research/MSK MRI/sample_data/meas_MID00333_FID118231_SAG_T2_FS.dat"
raw_file, kspace_np, info = import_kspace(filename)

In [ ]:
info

In [ ]:
kspace_real = np.real(kspace_np)
kspace_imag = np.imag(kspace_np)

# Step 2: stack into the last dimension -> (..., 2)
kspace_stack = np.stack((kspace_real, kspace_imag), axis=-1)

kspace = torch.from_numpy(kspace_stack).float()  # convert to torch tensor
kspace = kspace.permute(0, 1, 3, 2, 4, 5).contiguous()

kspace.shape  # torch.Size([n_part, n_line, n_channel, n_column

In [ ]:
from fastmri import fft2c, ifft2c, rss_complex, ifftshift, roll
import matplotlib.pyplot as plt

# For each slice in kspace, reconstruct and display the image
for slice_idx in range(kspace.shape[0]):
    kspace_slice = kspace[slice_idx]  # shape: [n_part, n_line, n_channel, n_column, 2]
    print(kspace_slice.shape)
    if info['readout_oversampling_factor'] > 1:
        nx = kspace_slice.shape[2]
        # Take the first nx//4 and the last nx//4 in the readout direction and stack them in the corresponding dimension
        kspace_slice = kspace_slice[:, :, 1::2, :, :]
        # kspace_slice = kspace_slice[:, :, :, 1::2, :]
    image = ifft2c(kspace_slice)  # shape: [n_part, n_line, n_channel, n_column, 2]
    rss_img = rss_complex(image, dim=1)  # Combine coils using root-sum-of-squares
    print(rss_img.shape)
    rss_img = roll(rss_img, [rss_img.shape[1]//2], dim=[1])  # Shift the image back
    img_np = rss_img.squeeze().cpu().numpy()
    
    # Also calculate the magnitude of the kspace of rss reconstruction for visualization. Do this by taking fft2c of rss_img
    # Add zeros as the imaginary dimensions since the image is real-valued
    # First, need to add back the coil dimension to rss_img
    rss_img_expanded = torch.stack((rss_img, torch.zeros_like(rss_img)), dim=-1)
    rss_img_expanded = rss_img_expanded.unsqueeze(1)  # shape: [n_part, 1, n_line, n_column, 2]
    kspace_rss = fft2c(rss_img_expanded)  # shape: [n_part, 1, n_line, n_column, 2]
    kspace_rss_magnitude = torch.log(1e-6 + torch.sqrt(kspace_rss[...,0]**2 + kspace_rss[...,1]**2).squeeze()).cpu().numpy()
    print(kspace_rss_magnitude.shape)
    
    # Split figure into 2 subplots: left for image, right for kspace magnitude
    fig, axs = plt.subplots(1, 2, figsize=(10, 5), dpi=200)
    axs[0].imshow(img_np, cmap='gray')
    axs[0].set_title(f"Reconstructed MRI Image - Slice {slice_idx+1}")
    axs[0].axis('off')
    axs[1].imshow(kspace_rss_magnitude, cmap='gray')
    axs[1].set_title(f"K-space Magnitude - Slice {slice_idx+1}")
    axs[1].axis('off')
    plt.show()

In [ ]:
data = twixtools.map_twix(filename)